# 05 — Sénat : socle depuis les données ouvertes

**Rôle :** constituer le socle Sénat depuis une source **ouverte et transparente** — Senate Stock Watcher (JSON sur GitHub). On utilise **`all_daily_summaries.json`** : il fournit, par dépôt, la **date de réception (= `disclosure_date`, anti look-ahead)** ET le **BioGuideID** du sénateur — deux champs absents de `all_transactions.json`.

**⚠️ Limite de fraîcheur (constatée à l'exécution) :** ce dataset ouvert s'arrête vers **mars 2021**. Pour la queue 2021→aujourd'hui, voir le secours **kadoa** (dernière cellule) ou la vérification eFD (notebook 06). Les lignes sans ticker (obligations/options) sont flaguées (**backlog**).

**Pas de scraping d'efdsearch ici** (Akamai + gate d'agrément) : on ne contourne aucune protection.

**Cellule 1 — Imports & chemins.**

In [ ]:
import requests, json
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
EXT_SENATE   = PROJECT_ROOT / "data" / "external" / "senate_openset"; EXT_SENATE.mkdir(parents=True, exist_ok=True)
PROC_SENATE  = PROJECT_ROOT / "data" / "processed" / "senate"; PROC_SENATE.mkdir(parents=True, exist_ok=True)
USER_AGENT   = "Ramify-QIS research (contact: <ton-email>)" 

**Cellule 2 — Téléchargement du dataset ouvert.** `all_daily_summaries.json` = résumés par dépôt, avec `date_recieved` et `bioguide`. Idempotent.

In [ ]:
SSW_URL = ("https://raw.githubusercontent.com/timothycarambat/"
           "senate-stock-watcher-data/master/aggregate/all_daily_summaries.json")
raw_path = EXT_SENATE / "ssw_all_daily_summaries.json"
if not raw_path.exists():
    r = requests.get(SSW_URL, headers={"User-Agent": USER_AGENT}, timeout=120)
    r.raise_for_status()
    raw_path.write_bytes(r.content)
summaries = json.loads(raw_path.read_text())
print("Dépôts (daily summaries) :", len(summaries))

**Cellule 3 — Aperçu d'un dépôt.** Un sénateur + `date_recieved` + une liste de transactions.

In [ ]:
{k: (f"<{len(v)} transactions>" if k == "transactions" else v) for k, v in summaries[0].items()}

**Cellule 4 — Aplatissement → une ligne par transaction.** On propage `date_recieved` (disclosure) et `bioguide` à chaque transaction.

In [ ]:
rows = []
for s in summaries:
    base = {"source": "senate", "provenance": "openset",
            "bioguide_id": s.get("bioguide"),
            "declarant_name": f'{s.get("first_name","")} {s.get("last_name","")}'.strip(),
            "disclosure_date": s.get("date_recieved"),
            "source_url": s.get("ptr_link")}
    for t in s.get("transactions", []):
        rows.append({**base,
            "transaction_date": t.get("transaction_date"), "owner": t.get("owner"),
            "ticker": t.get("ticker"), "asset_description": t.get("asset_description"),
            "asset_type": t.get("asset_type"), "operation_type": t.get("type"),
            "amount_range": t.get("amount")})
senate = pd.DataFrame(rows)
senate["ticker"] = senate["ticker"].replace({"--": None})
print("Transactions :", len(senate), "| sans ticker :", int(senate["ticker"].isna().sum()),
      "| avec BioGuideID :", int(senate["bioguide_id"].notna().sum()))

**Cellule 5 — Sauvegarde + couverture par an (disclosure).** Met en évidence la frontière de fraîcheur (~2021).

In [ ]:
senate["disclosure_date"] = pd.to_datetime(senate["disclosure_date"], errors="coerce")
senate.to_csv(PROC_SENATE / "senate_transactions.csv", index=False)
print(senate["disclosure_date"].dt.year.value_counts().sort_index())

**Cellule 6 — (Recommandé) Secours kadoa pour la queue récente.** Le socle s'arrêtant ~2021, complétez 2021→aujourd'hui avec le dataset MIT kadoa (« 2012→présent ») : renseignez l'URL du JSON et concaténez (même schéma, `provenance='openset-kadoa'`).

In [ ]:
# KADOA_URL = "https://raw.githubusercontent.com/kadoa-org/congress-trading-monitor/main/public/data/<fichier>.json"
# r = requests.get(KADOA_URL, headers={"User-Agent": USER_AGENT}, timeout=120); r.raise_for_status()
# (aplatir/normaliser comme ci-dessus avec provenance='openset-kadoa', puis concat avec `senate`)

Socle Sénat prêt ✅ (avec `disclosure_date` + BioGuideID) — vérification de provenance au **`06_senate_efd_provenance`**.